## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import re
import os
import glob
import json
from datetime import datetime
import warnings
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    pipeline
)
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.max_colwidth', 200)
pd.set_option('display.width', 1000)

# Check GPU availability
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


CUDA available: True
GPU: Tesla T4
GPU Memory: 15.64 GB


## Load Reviews Data

In [3]:
def load_reviews_data():
    """Load reviews data from CSV files"""
    reviews_df = pd.DataFrame()
    
    # Check multiple possible locations
    search_folders = ['/kaggle/input/datasets/tahmidakter/cleaned-review', '/kaggle/input/amazon-data', 
                      'cleaned_output', 'output', '.']
    
    for search_folder in search_folders:
        if os.path.exists(search_folder):
            # Find CSV files
            pattern = os.path.join(search_folder, '*.csv')
            csv_files = glob.glob(pattern)
            
            for file_path in csv_files:
                filename = os.path.basename(file_path).lower()
                try:
                    # Load only review files
                    if 'review' in filename:
                        df = pd.read_csv(file_path, encoding='utf-8')
                        if reviews_df.empty or len(df) > len(reviews_df):
                            reviews_df = df
                            print(f"Loaded reviews from: {file_path}")
                            print(f"Reviews shape: {df.shape}")
                            print(f"Columns: {list(df.columns)}")
                except Exception as e:
                    print(f"Error loading {file_path}: {e}")
            
            if not reviews_df.empty:
                break
    
    return reviews_df

# Load reviews
reviews_df = load_reviews_data()

if reviews_df.empty:
    print("\nNo review files found! Please upload your reviews CSV file.")
else:
    print(f"\nTotal reviews loaded: {len(reviews_df)}")

Loaded reviews from: /kaggle/input/datasets/tahmidakter/cleaned-review/cleaned_reviews_20260831_113113.csv
Reviews shape: (6648, 14)
Columns: ['review_title', 'review_rating', 'review_date', 'reviewer_name', 'verified_purchase', 'review_body', 'helpful_votes', 'scraped_at', 'product_id', 'asin', 'review_rating_numeric', 'review_date_parsed', 'helpful_votes_numeric', 'review_length']

Total reviews loaded: 6648


## Prepare Reviews for Analysis

In [4]:
def prepare_reviews(df):
    """Prepare reviews dataframe for FinBERT analysis"""
    
    if df.empty:
        return pd.DataFrame()
    
    df_prepared = df.copy()
    
    # Identify review text column
    text_columns = ['review_body', 'reviewText', 'review_text', 'body', 'text', 'review']
    text_col = None
    
    for col in text_columns:
        if col in df_prepared.columns:
            text_col = col
            break
    
    if text_col is None:
        print("No review text column found!")
        print("Available columns:", list(df_prepared.columns))
        return pd.DataFrame()
    
    print(f"Using '{text_col}' as review text column")
    
    # Rename to standard column
    df_prepared['review_text'] = df_prepared[text_col]
    
    # Clean review text
    df_prepared['review_text'] = df_prepared['review_text'].fillna('')
    df_prepared['review_text'] = df_prepared['review_text'].astype(str)
    df_prepared['review_text'] = df_prepared['review_text'].str.strip()
    
    # Remove empty reviews
    df_prepared = df_prepared[df_prepared['review_text'] != '']
    
    # Remove very short reviews (less than 3 words)
    df_prepared['word_count'] = df_prepared['review_text'].apply(lambda x: len(x.split()))
    df_prepared = df_prepared[df_prepared['word_count'] >= 3]
    
    # Truncate long reviews (FinBERT max length is 512 tokens)
    df_prepared['review_truncated'] = df_prepared['review_text'].apply(
        lambda x: ' '.join(x.split()[:400])
    )
    
    print(f"\nPrepared {len(df_prepared)} reviews for analysis")
    print(f"Average review length: {df_prepared['word_count'].mean():.2f} words")
    print(f"Max review length: {df_prepared['word_count'].max()} words")
    print(f"Min review length: {df_prepared['word_count'].min()} words")
    
    # Display sample reviews
    print("\n=== Sample Reviews ===")
    for i, review in enumerate(df_prepared['review_truncated'].head(5), 1):
        print(f"\nReview {i}: {review[:150]}...")
    
    return df_prepared

# Prepare reviews
reviews_prepared = prepare_reviews(reviews_df)


Using 'review_body' as review text column

Prepared 6342 reviews for analysis
Average review length: 54.20 words
Max review length: 992 words
Min review length: 3 words

=== Sample Reviews ===

Review 1: These are my favorite little guitars! I have small hands so I use them for me as an adult. I replace the strings with d'addario 1/2 silver strings...

Review 2: it looks and plays just like a regular guitar....

Review 3: Very disappointed! Item took longer to deliver than other similar guitars but we ordered it as the price was moderate and looked nice. Item did arrive...

Review 4: Perfect size for my 7 and 8 years kids. The sound quality is excellent....

Review 5: Had a similar issue as another reviewer where the neck looked like it was already separating from the body when it arrived. Originally I thought it wa...


## Load Model

In [29]:
class SentimentAnalyzer:
    """Sentiment Analysis wrapper for Amazon reviews"""

    def __init__(
        self,
        model_name="SebasLopez-ai/distilbert-amazon-reviews-sentiment",
        batch_size=32
    ):
        self.model_name = model_name
        self.batch_size = batch_size

        self.device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

        self.tokenizer = None
        self.model = None
        self.sentiment_pipeline = None

        print(f"Initializing model on {self.device}")

        self.load_model()

    def load_model(self):
        """Load model and tokenizer"""

        try:
            print(f"Loading tokenizer from {self.model_name}...")

            self.tokenizer = AutoTokenizer.from_pretrained(
                self.model_name
            )

            print(f"Loading model from {self.model_name}...")

            self.model = AutoModelForSequenceClassification.from_pretrained(
                self.model_name,

                # Your desired 3 sentiment classes
                num_labels=3,

                id2label={
                    0: "NEGATIVE",
                    1: "NEUTRAL",
                    2: "POSITIVE",
                },

                label2id={
                    "NEGATIVE": 0,
                    "NEUTRAL": 1,
                    "POSITIVE": 2,
                },

                # Original model has 5 labels
                ignore_mismatched_sizes=True,
            )

            # IMPORTANT:
            # Always move model to the selected device
            self.model = self.model.to(self.device)

            self.model.eval()

            print("Model loaded successfully!")
            print(f"Model device: {self.device}")

        except Exception as e:
            print(f"Error loading model: {e}")
            raise

    def create_pipeline(self):
        """Create sentiment analysis pipeline"""

        if self.sentiment_pipeline is None:

            self.sentiment_pipeline = pipeline(
                "sentiment-analysis",
                model=self.model,
                tokenizer=self.tokenizer,
                device=0 if self.device.type == "cuda" else -1,
                batch_size=self.batch_size
            )

        return self.sentiment_pipeline

    def predict_single(self, text):
        """Predict sentiment for a single review"""

        inputs = self.tokenizer(
            text,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        # Move tokenizer inputs to the SAME device as model
        inputs = {
            key: value.to(self.device)
            for key, value in inputs.items()
        }

        with torch.no_grad():

            outputs = self.model(**inputs)

            logits = outputs.logits

            probabilities = torch.softmax(
                logits,
                dim=-1
            )

        predicted_class = torch.argmax(
            probabilities,
            dim=-1
        ).item()

        confidence = probabilities[
            0,
            predicted_class
        ].item()

        return {
            "label": self.model.config.id2label[predicted_class],
            "confidence": confidence
        }

    def predict_batch(self, texts):
        """Predict sentiment for batch of texts"""

        results = []

        sentiment_pipeline = self.create_pipeline()

        for i in tqdm(
            range(0, len(texts), self.batch_size),
            desc="Analyzing reviews"
        ):

            batch_texts = texts[
                i:i + self.batch_size
            ]

            try:

                batch_results = sentiment_pipeline(
                    batch_texts
                )

                for result in batch_results:

                    results.append({
                        "label": result["label"],
                        "confidence": result["score"]
                    })

            except Exception as e:

                print(
                    f"\nError processing batch {i}: {e}"
                )

                # Fallback to single predictions
                for text in batch_texts:

                    try:

                        result = self.predict_single(text)

                        results.append(result)

                    except Exception as e2:

                        print(
                            f"Error predicting single text: {e2}"
                        )

                        results.append({
                            "label": "NEUTRAL",
                            "confidence": 0.0
                        })

        return results

analyzer = SentimentAnalyzer(batch_size=32)

Initializing model on cuda
Loading tokenizer from SebasLopez-ai/distilbert-amazon-reviews-sentiment...


config.json:   0%|          | 0.00/772 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/357 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loading model from SebasLopez-ai/distilbert-amazon-reviews-sentiment...


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully!
Model device: cuda


## Test with sample reviews

In [30]:
print("=== FinBERT Sentiment Analysis Demo ===\n")

sample_texts = [
    "This product is amazing! I love it so much, highly recommended!",
    "It's okay, does the job but nothing special.",
    "Terrible product, broke after one day. Waste of money!",
    "The quality is good but the price is too high.",
    "Absolutely perfect! Best purchase I've made this year!",
    "Not worth the money, very disappointed with the quality.",
    "Average product, works as expected but nothing extraordinary.",
    "Great value for money, would definitely buy again!"
]

print("Analyzing sample reviews...\n")
for text in sample_texts:

    result = analyzer.predict_single(text)

    print(f"Text: {text}")
    print(f"  Sentiment: {result['label']}")
    print(f"  Confidence: {result['confidence']:.3f}")
    print()

=== FinBERT Sentiment Analysis Demo ===

Analyzing sample reviews...

Text: This product is amazing! I love it so much, highly recommended!
  Sentiment: POSITIVE
  Confidence: 0.997

Text: It's okay, does the job but nothing special.
  Sentiment: NEUTRAL
  Confidence: 0.939

Text: Terrible product, broke after one day. Waste of money!
  Sentiment: NEGATIVE
  Confidence: 0.995

Text: The quality is good but the price is too high.
  Sentiment: NEUTRAL
  Confidence: 0.734

Text: Absolutely perfect! Best purchase I've made this year!
  Sentiment: POSITIVE
  Confidence: 0.997

Text: Not worth the money, very disappointed with the quality.
  Sentiment: NEGATIVE
  Confidence: 0.952

Text: Average product, works as expected but nothing extraordinary.
  Sentiment: NEUTRAL
  Confidence: 0.944

Text: Great value for money, would definitely buy again!
  Sentiment: POSITIVE
  Confidence: 0.994



## Apply to All Reviews

In [31]:
def apply_sentiment(df, analyzer):
    """Apply sentiment analysis to all reviews"""
    
    if df.empty:
        return df
    
    df_result = df.copy()
    
    print(f"Analyzing {len(df_result)} reviews...")
    print(f"Batch size: {analyzer.batch_size}")
    print(f"Device: {analyzer.device}")
    
    # Get all review texts
    texts = df_result['review_truncated'].tolist()
    
    # Predict sentiments
    print("\nRunning sentiment analysis...")
    results = analyzer.predict_batch(texts)
    
    # Add results to dataframe
    df_result['sentiment'] = [r['label'] for r in results]
    df_result['confidence'] = [r['confidence'] for r in results]
    
    # Map labels to numeric scores
    label_to_num = {'positive': 1, 'neutral': 0, 'negative': -1}
    df_result['sentiment_score'] = df_result['sentiment'].map(label_to_num)
    
    # Print distribution
    print("\n=== Sentiment Distribution ===")
    sentiment_counts = df_result['sentiment'].value_counts()
    print(sentiment_counts)
    print(f"\nPercentage distribution:")
    print((sentiment_counts / len(df_result) * 100).round(2))
    
    return df_result

# Apply FinBERT sentiment analysis
if not reviews_prepared.empty:
    print("\n" + "="*60)
    print("Applying Sentiment Analysis...")
    print("="*60)
    reviews_sentiment = apply_sentiment(reviews_prepared, analyzer)
else:
    reviews_sentiment = pd.DataFrame()
    print("No reviews to analyze!")


Applying FinBERT Sentiment Analysis...
Analyzing 6342 reviews...
Batch size: 32
Device: cuda

Running sentiment analysis...


Analyzing reviews:   4%|▎         | 7/199 [00:02<01:05,  2.93it/s]Token indices sequence length is longer than the specified maximum sequence length for this model (521 > 512). Running this sequence through the model will result in indexing errors



Error processing batch 224: The size of tensor a (521) must match the size of tensor b (512) at non-singleton dimension 1


Analyzing reviews:  49%|████▉     | 98/199 [00:29<00:29,  3.45it/s]


Error processing batch 3104: The size of tensor a (522) must match the size of tensor b (512) at non-singleton dimension 1


...


Error processing batch 5728: The size of tensor a (530) must match the size of tensor b (512) at non-singleton dimension 1


Analyzing reviews:  93%|█████████▎| 185/199 [00:57<00:04,  3.23it/s]


Error processing batch 5888: The size of tensor a (569) must match the size of tensor b (512) at non-singleton dimension 1


Analyzing reviews: 100%|██████████| 199/199 [01:01<00:00,  3.25it/s]


=== Sentiment Distribution ===
sentiment
POSITIVE    4906
NEGATIVE     767
NEUTRAL      669
Name: count, dtype: int64

Percentage distribution:
sentiment
POSITIVE    77.36
NEGATIVE    12.09
NEUTRAL     10.55
Name: count, dtype: float64


# Convert ratings to sentiment labels (if not already done)

In [35]:
print(reviews_sentiment['sentiment'].value_counts())

sentiment
POSITIVE    4906
NEGATIVE     767
NEUTRAL      669
Name: count, dtype: int64


## Train test split


In [36]:
from sklearn.model_selection import train_test_split

train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_finetune['review_truncated'].tolist(),
    df_finetune['label_id'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df_finetune['label_id']  # preserve class distribution
)
print(f"Training samples: {len(train_texts)}, Validation samples: {len(val_texts)}")

Training samples: 5073, Validation samples: 1269


## Tokenization

In [37]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("SebasLopez-ai/distilbert-amazon-reviews-sentiment")

def tokenize_function(texts, labels):
    encodings = tokenizer(
        texts,
        truncation=True,
        padding=True,
        max_length=512,
        return_tensors=None  # We'll wrap in Dataset later
    )
    encodings['labels'] = labels
    return encodings

train_encodings = tokenize_function(train_texts, train_labels)
val_encodings = tokenize_function(val_texts, val_labels)

## Review Dataset

In [38]:
import torch
from torch.utils.data import Dataset

class ReviewDataset(Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        return item

    def __len__(self):
        return len(self.encodings['input_ids'])

train_dataset = ReviewDataset(train_encodings)
val_dataset = ReviewDataset(val_encodings)

## Load Model


In [39]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "SebasLopez-ai/distilbert-amazon-reviews-sentiment",
    num_labels=3,
    id2label={0: "NEGATIVE", 1: "NEUTRAL", 2: "POSITIVE"},
    label2id={"NEGATIVE": 0, "NEUTRAL": 1, "POSITIVE": 2},
    ignore_mismatched_sizes=True
)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

## Set up training arguments and Trainer

In [42]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./finetuned_amazon_sentiment",
    num_train_epochs=3,              # adjust as needed
    per_device_train_batch_size=16,  # adjust based on GPU memory
    per_device_eval_batch_size=32,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
)

# Define accuracy metric
from sklearn.metrics import accuracy_score

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions.argmax(axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## Fine‑tune and save

In [43]:
trainer.train()

# Save the final model
model.save_pretrained("./finetuned_model")
tokenizer.save_pretrained("./finetuned_model")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.528031,0.430058,0.929078
2,0.388505,0.475235,0.921198
3,0.323488,0.466475,0.925926


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./finetuned_model/tokenizer_config.json', './finetuned_model/tokenizer.json')

## Evaluate on Validation set

In [44]:
eval_results = trainer.evaluate()
print(f"Validation accuracy: {eval_results['eval_accuracy']:.4f}")

Validation accuracy: 0.9291


In [46]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report
import numpy as np

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    # 1. Overall Accuracy
    acc = accuracy_score(labels, predictions)
    
    # 2. Weighted averages (accounts for class imbalance)
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        labels, predictions, average='weighted'
    )
    
    # 3. Macro averages (treats all classes equally, better for imbalanced data)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, predictions, average='macro'
    )
    
    # 4. Per-class metrics (NEGATIVE, NEUTRAL, POSITIVE)
    precision_per, recall_per, f1_per, support_per = precision_recall_fscore_support(
        labels, predictions, average=None, labels=[0, 1, 2]
    )
    
    return {
        # Overall
        'accuracy': acc,
        
        # Aggregated
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted,
        
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        
        # Per-class (for detailed debugging)
        'precision_neg': precision_per[0],
        'recall_neg': recall_per[0],
        'f1_neg': f1_per[0],
        
        'precision_neu': precision_per[1],
        'recall_neu': recall_per[1],
        'f1_neu': f1_per[1],
        
        'precision_pos': precision_per[2],
        'recall_pos': recall_per[2],
        'f1_pos': f1_per[2],
    }

In [47]:
# After trainer.train() and trainer.evaluate()
predictions = trainer.predict(val_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)
true_labels = predictions.label_ids

# Generate the detailed text report
report = classification_report(
    true_labels, 
    pred_labels, 
    target_names=['NEGATIVE', 'NEUTRAL', 'POSITIVE'],
    digits=4
)
print("\n=== Detailed Classification Report ===")
print(report)


=== Detailed Classification Report ===
              precision    recall  f1-score   support

    NEGATIVE     0.6545    0.5806    0.6154        62
     NEUTRAL     0.3721    0.3556    0.3636        45
    POSITIVE     0.9624    0.9699    0.9661      1162

    accuracy                         0.9291      1269
   macro avg     0.6630    0.6354    0.6484      1269
weighted avg     0.9264    0.9291    0.9276      1269



In [48]:
import shutil

shutil.make_archive('finetuned_model', 'zip', './finetuned_model')
print("✅ Model zipped successfully as 'finetuned_model.zip'")

✅ Model zipped successfully as 'finetuned_model.zip'
